In [1]:
!pip install -U "transformers>=4.44" "peft>=0.13" "trl>=0.11" \
                   "bitsandbytes>=0.43" "accelerate>=0.34" "datasets>=3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer

In [3]:
# ---------------------------------------------------------------------------
# 1. Config
# ---------------------------------------------------------------------------
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama-qlora-adapter"
USE_QLORA = True   # set False for plain LoRA (fp16 base, ~2x more VRAM)

In [4]:
# ---------------------------------------------------------------------------
# 2. Tiny instruction dataset
# ---------------------------------------------------------------------------
# A real run uses something like tatsu-lab/alpaca, OpenAssistant/oasst1, or
# a domain dataset. Here we use 12 hand-written examples just to show the
# pipeline; the model will overfit, which is exactly what we want to verify
# end-to-end correctness.
raw_examples = [
    {"instruction": "What is the capital of France?",
     "response": "The capital of France is Paris."},
    {"instruction": "Define machine learning in one sentence.",
     "response": "Machine learning is the study of algorithms that improve their performance on a task through exposure to data."},
    {"instruction": "What does LoRA stand for?",
     "response": "LoRA stands for Low-Rank Adaptation, a parameter-efficient fine-tuning technique."},
    {"instruction": "What does QLoRA add to LoRA?",
     "response": "QLoRA quantizes the frozen base model to 4-bit (NF4) so LoRA adapters can be trained on much larger models with the same GPU memory."},
    {"instruction": "Explain attention in transformers briefly.",
     "response": "Attention computes a weighted sum of value vectors, where weights come from the similarity between query and key vectors."},
    {"instruction": "What is gradient checkpointing?",
     "response": "Gradient checkpointing trades compute for memory by recomputing activations during the backward pass instead of storing them."},
    {"instruction": "Name two parameter-efficient fine-tuning methods.",
     "response": "Two common methods are LoRA (Low-Rank Adaptation) and prefix tuning."},
    {"instruction": "What is the role of the tokenizer?",
     "response": "The tokenizer converts raw text into integer token IDs that the model can process, and decodes IDs back into text."},
    {"instruction": "Why use 4-bit NF4 quantization?",
     "response": "NF4 is information-theoretically optimal for normally-distributed weights, preserving more accuracy than naive 4-bit quantization."},
    {"instruction": "What is a LoRA rank?",
     "response": "The LoRA rank r is the inner dimension of the two low-rank matrices A and B; smaller r means fewer trainable parameters."},
    {"instruction": "What is supervised fine-tuning?",
     "response": "Supervised fine-tuning trains a pretrained model on labeled input/output pairs to specialize it for a downstream task."},
    {"instruction": "When should I prefer full fine-tuning over LoRA?",
     "response": "Prefer full fine-tuning when you have abundant compute and need to shift the model's behavior substantially across many capabilities."},
]


In [5]:
# ---------------------------------------------------------------------------
# 3. Tokenizer + chat-template formatting
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(ex):
    """Apply the model's chat template to produce a single training string."""
    messages = [
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(raw_examples).map(format_example)
print(f"Dataset size: {len(dataset)}")
print("Example:\n", dataset[0]["text"][:300], "...\n")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Dataset size: 12
Example:
 <|user|>
What is the capital of France?</s>
<|assistant|>
The capital of France is Paris.</s>
 ...



In [6]:
# ---------------------------------------------------------------------------
# 4. Load base model (QLoRA = 4-bit NF4)
# ---------------------------------------------------------------------------
if USE_QLORA and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",            # NF4 > FP4 for normal weights
        bnb_4bit_compute_dtype=torch.bfloat16, # matmul in bf16, weights in 4-bit
        bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model)  # casts norms, enables grad ckpt hooks
else:
    # Plain LoRA fallback (or CPU smoke test)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype)
    if torch.cuda.is_available():
        model = model.to("cuda")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
# ---------------------------------------------------------------------------
# 5. LoRA config + attach adapters
# ---------------------------------------------------------------------------
# target_modules: the projections we inject low-rank updates into.
# For Llama-family models these are the attention + MLP projections.
# Increasing r and alpha = more capacity but more params to train.
lora_config = LoraConfig(
    r=16,                       # rank
    lora_alpha=32,              # scaling = alpha / r = 2.0
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect something like: trainable params: 18M || all params: 1.1B || trainable%: 1.6

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [11]:
# ---------------------------------------------------------------------------
# 6. Training
# ---------------------------------------------------------------------------
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # effective batch size = 8
    learning_rate=2e-4,               # LoRA tolerates higher LR than full FT
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if USE_QLORA and torch.cuda.is_available() else "adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=512,                   # TRL >= 0.20: renamed from max_seq_length
    # dataset_text_field is auto-detected when the column is named "text".
    # If your column has a different name, pass it as a kwarg to SFTTrainer.
    report_to="none",                 # set to "wandb" / "tensorboard" if desired
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,       # TRL >= 0.16: replaces `tokenizer=`
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,3.298455
2,3.835165
3,2.716218
4,2.076623
5,2.065019
6,1.512367


TrainOutput(global_step=6, training_loss=2.583974599838257, metrics={'train_runtime': 25.8089, 'train_samples_per_second': 1.395, 'train_steps_per_second': 0.232, 'total_flos': 12955068407808.0, 'train_loss': 2.583974599838257})

In [12]:
# ---------------------------------------------------------------------------
# 7. Save the adapter (NOT the full model — just the LoRA weights, ~70MB)
# ---------------------------------------------------------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nAdapter saved to: {OUTPUT_DIR}")


Adapter saved to: ./tinyllama-qlora-adapter


In [10]:
!pip install -U "transformers>=4.46" "trl>=0.20" "peft>=0.13" \
               "bitsandbytes>=0.43" "accelerate>=1.0" "datasets>=3.0"

In [13]:
# ---------------------------------------------------------------------------
# 8. Reload base + adapter for inference
# ---------------------------------------------------------------------------
# Free the trainer's references first to avoid VRAM thrashing
del model, trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

if USE_QLORA and torch.cuda.is_available():
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    if torch.cuda.is_available():
        base = base.to("cuda")

ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft_model.eval()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [15]:
# ---------------------------------------------------------------------------
# 9. Inference
# ---------------------------------------------------------------------------
def generate(prompt: str, max_new_tokens: int = 128) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(ft_model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


print("\n=== Inference on a held-out prompt ===")
print(generate("What does QLoRA add to LoRA?"))

{"instruction": "What does QLoRA add to LoRA?",
     "response": "QLoRA quantizes the frozen base model to 4-bit (NF4) so LoRA adapters can be trained on much larger models with the same GPU memory."},
print("\n=== Inference on an unseen prompt ===")
print(generate("Explain the purpose of LoRA adapters in two sentences."))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== Inference on a held-out prompt ===


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QLoRA adds to LoRA by providing a new approach to the LoRA protocol that improves the security and privacy of the network.

=== Inference on an unseen prompt ===
LoRA adapters are used to connect LoRA-compatible devices to LoRaWAN networks.


In [18]:
#---------------------------------------------------------------------------
# 10. (Optional) Merge adapter into base weights for deployment
# ---------------------------------------------------------------------------
# Merging only works on the un-quantized base model. For QLoRA, you typically
# either (a) keep the adapter separate and load it at runtime, or (b) reload
# the base in fp16/bf16, merge, and save the full merged model:
#
base_fp16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base_fp16, OUTPUT_DIR).merge_and_unload()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported